# Project 29 — Citation Verifier for RAG
## Check if LLM Answers Are Supported by Retrieved Chunks

**Stack:** Ollama · LangChain · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

## Step 2 — Test Cases: Answers + Source Evidence

In [ ]:
test_cases = [
    {
        "question": "What is the capital of France?",
        "answer": "The capital of France is Paris. It has a population of about 2.1 million.",
        "sources": ["Paris is the capital and most populous city of France, with a population of 2,102,650."],
        "expected": "supported",
    },
    {
        "question": "When was Python created?",
        "answer": "Python was created in 1991 by Guido van Rossum. It was designed at Microsoft.",
        "sources": ["Python was conceived in the late 1980s by Guido van Rossum at CWI in the Netherlands."],
        "expected": "partially_supported",  # year close but 'Microsoft' is wrong
    },
    {
        "question": "How fast is GPT-4?",
        "answer": "GPT-4 can process 1 million tokens per second and costs $0.001 per query.",
        "sources": ["GPT-4 is a large language model created by OpenAI."],
        "expected": "not_supported",
    },
]
print(f"Prepared {len(test_cases)} test cases for verification")

## Step 3 — Claim Extraction

In [ ]:
class ExtractedClaims(BaseModel):
    claims: list[str] = Field(description="Individual factual claims from the answer")

claim_extractor = llm.with_structured_output(ExtractedClaims)

for tc in test_cases:
    claims = claim_extractor.invoke(
        f"Extract individual factual claims from this answer:\n\n{tc['answer']}"
    )
    tc["claims"] = claims.claims
    print(f"Q: {tc['question']}")
    print(f"  Claims extracted: {len(claims.claims)}")
    for c in claims.claims:
        print(f"    • {c}")
    print()

## Step 4 — Verify Each Claim Against Sources

In [ ]:
class ClaimVerification(BaseModel):
    claim: str
    verdict: str = Field(description="supported, contradicted, or not_mentioned")
    evidence: str = Field(description="Quote from source supporting/contradicting, or 'none'")
    confidence: float = Field(description="0.0 to 1.0")

class VerificationReport(BaseModel):
    claim_results: list[ClaimVerification]
    overall_verdict: str = Field(description="supported, partially_supported, not_supported")
    groundedness_score: float = Field(description="0.0 to 1.0 — fraction of supported claims")

verifier = llm.with_structured_output(VerificationReport)

for tc in test_cases:
    sources_text = "\n".join(tc["sources"])
    claims_text = "\n".join([f"- {c}" for c in tc["claims"]])

    report = verifier.invoke(
        f"""Verify each claim against the source evidence.

Claims:
{claims_text}

Source Evidence:
{sources_text}

For each claim, determine if it is supported, contradicted, or not mentioned
in the sources."""
    )

    print(f"\nQ: {tc['question']}")
    print(f"Answer: {tc['answer']}")
    print(f"Verdict: {report.overall_verdict} (expected: {tc['expected']})")
    print(f"Groundedness: {report.groundedness_score:.0%}")
    for cr in report.claim_results:
        icon = {"supported": "✓", "contradicted": "✗", "not_mentioned": "?"}
        print(f"  {icon.get(cr.verdict, '?')} [{cr.verdict}] {cr.claim}")
        if cr.evidence != "none":
            print(f"    Evidence: {cr.evidence[:80]}...")

## What You Learned
- **Claim extraction** from generated answers
- **Evidence-based verification** of individual claims
- **Groundedness scoring** for RAG quality assessment
- **Distinguishing** supported, contradicted, and unsupported claims